# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Deployable no-context validation
# ------------------------------
# The competition test file has no prompt_id. This experiment therefore does
# not use same-prompt peers at inference. Style features are normalized only
# with statistics fitted on the training portion of each fold.
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score

OUTPUT_PATH = Path('/kaggle/working/no_context_results.json')
BEST_CHAR_RANGE = (1, 5)
BEST_C = 2.0

FEATURE_NAMES = [
    'characters', 'words', 'sentences', 'punctuation', 'digits', 'uppercase',
    'question_marks', 'exclamation_marks', 'colons', 'semicolons',
    'brackets', 'markdown_markers', 'bullet_markers', 'newline_count',
]

def response_pattern(text):
    words = re.findall(r"\b[\w']+\b", text)
    return [
        float(len(text)), float(len(words)),
        float(len(re.findall(r'[.!?]+', text))),
        float(sum(text.count(c) for c in PUNCTUATION)),
        float(sum(c.isdigit() for c in text)),
        float(sum(c.isupper() for c in text)),
        float(text.count('?')), float(text.count('!')),
        float(text.count(':')), float(text.count(';')),
        float(sum(text.count(c) for c in '()[]{}')),
        float(sum(text.count(c) for c in '#*_`')),
        float(sum(text.count(c) for c in '*-•')),
        float(text.count('\\n')),
    ]

style = np.asarray([response_pattern(text) for text in texts], dtype=np.float32)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
results = {'tfidf_only': [], 'tfidf_plus_global_style': []}
fold_records = []
started = time.time()

for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
    word_vectorizer = make_word_vectorizer()
    X_word_train = word_vectorizer.fit_transform(texts[train_idx])
    X_word_valid = word_vectorizer.transform(texts[valid_idx])

    punctuation_scaler = StandardScaler()
    X_punct_train = csr_matrix(punctuation_scaler.fit_transform(punctuation[train_idx]))
    X_punct_valid = csr_matrix(punctuation_scaler.transform(punctuation[valid_idx]))

    char_vectorizer = make_char_vectorizer(BEST_CHAR_RANGE)
    X_char_train = char_vectorizer.fit_transform(texts[train_idx])
    X_char_valid = char_vectorizer.transform(texts[valid_idx])

    X_text_train = hstack([X_word_train, X_char_train, X_punct_train], format='csr')
    X_text_valid = hstack([X_word_valid, X_char_valid, X_punct_valid], format='csr')

    def train_score(X_train, X_valid):
        model = LinearSVC(C=BEST_C, class_weight='balanced', max_iter=2_000)
        model.fit(X_train, labels[train_idx])
        return float(balanced_accuracy_score(labels[valid_idx], model.predict(X_valid)))

    baseline_score = train_score(X_text_train, X_text_valid)
    results['tfidf_only'].append(baseline_score)

    # Strictly deployable: fit style normalization on training rows only.
    style_scaler = StandardScaler()
    style_train = csr_matrix(style_scaler.fit_transform(style[train_idx]))
    style_valid = csr_matrix(style_scaler.transform(style[valid_idx]))
    combined_train = hstack([X_text_train, style_train], format='csr')
    combined_valid = hstack([X_text_valid, style_valid], format='csr')
    style_score = train_score(combined_train, combined_valid)
    results['tfidf_plus_global_style'].append(style_score)

    fold_records.append({'fold': fold_number, 'tfidf_only': baseline_score, 'tfidf_plus_global_style': style_score})
    print(f'Fold {fold_number}/{N_SPLITS}: tfidf={baseline_score:.4f} | no_context_style={style_score:.4f}')

summary_results = {}
for name, scores in results.items():
    summary_results[name] = {
        'mean_balanced_accuracy': float(np.mean(scores)),
        'std_balanced_accuracy': float(np.std(scores, ddof=1)),
        'fold_scores': scores,
    }
ranking = sorted(summary_results.items(), key=lambda item: item[1]['mean_balanced_accuracy'], reverse=True)
summary = {
    'experiment': 'deployable no-context style features',
    'split_method': '5-fold GroupKFold by prompt_id',
    'inference_context': 'text only; no prompt_id or same-prompt peers required',
    'text_config': {'word_ngram_range': list(WORD_NGRAM_RANGE), 'char_ngram_range': list(BEST_CHAR_RANGE), 'linear_svc_C': BEST_C},
    'style_feature_names': FEATURE_NAMES,
    'results': summary_results,
    'fold_records': fold_records,
    'ranking': [{'configuration': key, **value} for key, value in ranking],
    'total_elapsed_seconds': round(time.time() - started, 2),
}
OUTPUT_PATH.write_text(json.dumps(summary, indent=2))
print('\\nFinal ranking:')
for position, (key, value) in enumerate(ranking, start=1):
    print(f'{position}. {key}: {value["mean_balanced_accuracy"]:.4f} +/- {value["std_balanced_accuracy"]:.4f}')
print(f'Saved results to {OUTPUT_PATH}')


## Deployability decision

Only a model that does not require `prompt_id` or same-prompt peers can be used directly on the competition test file. The no-context result is the correct score to use for submission planning.
